In [ ]:
import os
import glob
import shutil
import yaml
import random
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ==============================================================
# CONFIGURATION — Edit paths to match your Kaggle input
# ==============================================================

DATASETS = {
    "weapon1": {
        "root": "/kaggle/input/datasets/mfonisovic/weapon1",
        "splits": {
            "train": ("train/images", "train/labels"),
            "val":   ("valid/images", "valid/labels"),
            "test":  ("test/images",  "test/labels"),
        },
        # Original class 0 = person -> skip (None)
        # Original class 1 = generic weapon -> map to unified 0 (weapon)
        "class_map": {0: None, 1: 0},
    },
    "weapons2": {
        "root": "/kaggle/input/datasets/mfonisovic/weapons2",
        "splits": {
            "train": ("train/images", "train/labels"),
        },
        # All classes are weapons -> map all to unified 0
        "class_map": {0: 0, 1: 0, 2: 0, 3: 0},   # grenade, rifle, knife, pistol -> weapon
    },
    "weapons3": {
        "root": "/kaggle/input/datasets/mfonisovic/weapons-3",
        "splits": {
            "train": ("train/images",  "train/labels"),
            "val":   ("valid/images",  "valid/labels"),
            "test":  ("test/images",   "test/labels"),
        },
        # All classes are weapons -> map to 0
        "class_map": {0: 0, 1: 0, 2: 0},   # handgun, rifle, knife -> weapon
    },
    "weapon_detection_dataset": {
        "root": "/kaggle/input/datasets/snehilsanyal/weapon-detection-test/weapon_detection",
        "splits": {
            "train": ("train/images", "train/labels"),
            "val":   ("val/images",   "val/labels"),
        },
        # metadata.csv defines train/val split
        "metadata_csv": "/kaggle/input/weapon-detection-dataset/metadata.csv",
        # All 9 classes are weapons -> map all to 0
        "class_map": {i: 0 for i in range(9)},
    },
}

OUTPUT_DIR = "/kaggle/working/unified_dataset_one_class"
TRAIN_RATIO = 0.80   # used only for datasets without splits
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
SEED        = 42

# Unified class list (single class)
UNIFIED_CLASSES = ["weapon"]

# ==============================================================
# SETUP OUTPUT DIRECTORIES
# ==============================================================
for split in ["train", "val", "test"]:
    for sub in ["images", "labels"]:
        Path(f"{OUTPUT_DIR}/{split}/{sub}").mkdir(parents=True, exist_ok=True)

print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📋 Unified classes: {UNIFIED_CLASSES}\n")

# ==============================================================
# HELPERS
# ==============================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_image(images_dir: str, stem: str) -> str | None:
    for ext in IMAGE_EXTS:
        p = os.path.join(images_dir, stem + ext)
        if os.path.exists(p):
            return p
    return None

def remap_label_file(src_label: str, class_map: dict) -> list[str] | None:
    """Read YOLO label, remap class IDs, return new lines."""
    with open(src_label, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    new_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) < 5:
            continue
        orig_cls = int(parts[0])
        new_cls = class_map.get(orig_cls, None)
        if new_cls is None:
            continue   # skip this annotation (e.g., person)
        new_lines.append(f"{new_cls} " + " ".join(parts[1:]))
    return new_lines if new_lines else None

def safe_copy(src: str, dst: str):
    """Copy file, avoid collisions."""
    if os.path.exists(dst):
        base, ext = os.path.splitext(dst)
        dst = f"{base}_dup{ext}"
    shutil.copy2(src, dst)

def add_to_split(split: str, ds_prefix: str,
                 img_src: str, label_lines: list[str]):
    stem = os.path.splitext(os.path.basename(img_src))[0]
    ext  = os.path.splitext(img_src)[1]
    new_name = f"{ds_prefix}__{stem}"

    dst_img = os.path.join(OUTPUT_DIR, split, "images", new_name + ext)
    safe_copy(img_src, dst_img)

    dst_lbl = os.path.join(OUTPUT_DIR, split, "labels", new_name + ".txt")
    with open(dst_lbl, "w") as f:
        f.write("\n".join(label_lines) + "\n")

stats = defaultdict(lambda: defaultdict(int))

# ==============================================================
# PROCESS EACH DATASET
# ==============================================================
for ds_name, cfg in DATASETS.items():
    root      = cfg["root"]
    class_map = cfg["class_map"]
    print(f"{'='*60}")
    print(f"  Processing: {ds_name}")
    print(f"  Class map : {class_map}")
    print(f"{'='*60}")

    # Special handling for weapon_detection_dataset (uses metadata.csv)
    if "metadata_csv" in cfg:
        meta_path = cfg["metadata_csv"]
        if os.path.exists(meta_path):
            meta = pd.read_csv(meta_path)
            print(f"  Loaded metadata.csv — {len(meta)} entries")
            for _, row in meta.iterrows():
                img_file   = row["imagefile"]
                label_file = row["labelfile"]
                split_flag = int(row["train_id"])   # 1=train, 0=val
                split      = "train" if split_flag == 1 else "val"

                img_src = lbl_src = None
                for split_dir in ["train", "val"]:
                    img_candidate = os.path.join(root, split_dir, "images", img_file)
                    lbl_candidate = os.path.join(root, split_dir, "labels", label_file)
                    if os.path.exists(img_candidate) and os.path.exists(lbl_candidate):
                        img_src = img_candidate
                        lbl_src = lbl_candidate
                        break

                if img_src is None or lbl_src is None:
                    continue
                new_lines = remap_label_file(lbl_src, class_map)
                if new_lines is None:
                    continue
                add_to_split(split, ds_name, img_src, new_lines)
                stats[ds_name][split] += 1
        else:
            print(f"  [WARN] metadata.csv not found at {meta_path}")
        continue   # done with this dataset

    # Standard processing: iterate explicit split folders
    available_splits = {}
    for split_name, (img_rel, lbl_rel) in cfg["splits"].items():
        img_dir = os.path.join(root, img_rel)
        lbl_dir = os.path.join(root, lbl_rel)
        if os.path.isdir(lbl_dir) and os.path.isdir(img_dir):
            available_splits[split_name] = (img_dir, lbl_dir)
        else:
            print(f"  [INFO] Split '{split_name}' not found — skipping")

    if not available_splits:
        print(f"  [WARN] No valid splits found for {ds_name}. Skipping.")
        continue

    for split_name, (img_dir, lbl_dir) in available_splits.items():
        label_files = glob.glob(os.path.join(lbl_dir, "*.txt"))
        print(f"  [{split_name}] {len(label_files)} label files found")
        for lf in label_files:
            stem = os.path.splitext(os.path.basename(lf))[0]
            img_src = find_image(img_dir, stem)
            if img_src is None:
                continue
            new_lines = remap_label_file(lf, class_map)
            if new_lines is None:
                continue
            add_to_split(split_name, ds_name, img_src, new_lines)
            stats[ds_name][split_name] += 1

# ==============================================================
# AUTO‑SPLIT DATASETS THAT ONLY HAVE TRAIN (e.g., weapons2)
# ==============================================================
random.seed(SEED)

def auto_resplit(ds_prefix: str, train_dir_images: str, train_dir_labels: str):
    all_imgs = glob.glob(os.path.join(train_dir_images, f"{ds_prefix}__*"))
    random.shuffle(all_imgs)
    n = len(all_imgs)
    n_val  = int(n * VAL_RATIO)
    n_test = int(n * TEST_RATIO)

    val_imgs  = all_imgs[:n_val]
    test_imgs = all_imgs[n_val:n_val + n_test]

    moved = {"val": 0, "test": 0}
    for split, imgs in [("val", val_imgs), ("test", test_imgs)]:
        for img_path in imgs:
            fname = os.path.basename(img_path)
            stem  = os.path.splitext(fname)[0]
            ext   = os.path.splitext(fname)[1]
            lbl_src = os.path.join(train_dir_labels, stem + ".txt")
            if not os.path.exists(lbl_src):
                continue
            dst_img = os.path.join(OUTPUT_DIR, split, "images", fname)
            shutil.move(img_path, dst_img)
            dst_lbl = os.path.join(OUTPUT_DIR, split, "labels", stem + ".txt")
            shutil.move(lbl_src, dst_lbl)
            moved[split] += 1
    return moved

print(f"\n{'='*60}")
print("  Auto‑splitting train‑only datasets (weapons2)...")
print(f"{'='*60}")
moved = auto_resplit(
    "weapons2",
    os.path.join(OUTPUT_DIR, "train", "images"),
    os.path.join(OUTPUT_DIR, "train", "labels"),
)
print(f"  weapons2 — moved {moved['val']} to val, {moved['test']} to test")
stats["weapons2"]["val"]  += moved["val"]
stats["weapons2"]["test"] += moved["test"]
stats["weapons2"]["train"] = max(0, stats["weapons2"].get("train", 0) - moved["val"] - moved["test"])

# ==============================================================
# GENERATE data.yaml
# ==============================================================
yaml_content = {
    "path":  OUTPUT_DIR,
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(UNIFIED_CLASSES),
    "names": UNIFIED_CLASSES,
}
yaml_out = os.path.join(OUTPUT_DIR, "data.yaml")
with open(yaml_out, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
print(f"\n✅ data.yaml written to: {yaml_out}")

# ==============================================================
# FINAL SUMMARY
# ==============================================================
print(f"\n{'='*60}")
print("  ✅ MERGE COMPLETE")
print(f"{'='*60}")
total = defaultdict(int)
for ds_name, split_counts in stats.items():
    print(f"\n  📦 {ds_name}")
    for split_name, count in sorted(split_counts.items()):
        print(f"     {split_name:10s}: {count:>5} samples")
        total[split_name] += count

print(f"\n  {'─'*40}")
print(f"  TOTAL")
for split_name, count in sorted(total.items()):
    print(f"     {split_name:10s}: {count:>5} samples")
print(f"  {'─'*40}")
print(f"  Grand total : {sum(total.values())} samples")

# Class distribution (only one class)
print(f"\n  📊 Class distribution (all weapon instances):")
lbl_dir = os.path.join(OUTPUT_DIR, "train", "labels")
total_instances = 0
for lf in glob.glob(os.path.join(lbl_dir, "*.txt")):
    with open(lf) as f:
        total_instances += sum(1 for _ in f)
print(f"     weapons: {total_instances} instances")
print(f"\n  To start training (single‑class YOLO):")
print(f"     from ultralytics import YOLO")
print(f"     model = YOLO('yolov10s.pt')")
print(f"     model.train(data='{yaml_out}', epochs=100, imgsz=640, single_cls=True)")

In [ ]:
# %% [markdown]
# # 🚀 Weapon Detection with YOLOv10s – Single Class "weapon"
# 
# This notebook trains a YOLOv10s model on the unified one‑class weapon dataset.
# 
# **Dataset:** Merged dataset at `/kaggle/working/unified_dataset_one_class`  
# **Model:** YOLOv10s (pre‑trained on COCO)  
# **Classes:** 1 (weapon)
# 
# ---

# %% [code]
# ─── 1. Install Dependencies ──────────────────────────────────────────────────
!pip install -q ultralytics

import os
import yaml
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO

# %% [code]
# ─── 2. Configuration ────────────────────────────────────────────────────────
# Path to the unified dataset (created by the merging notebook)
DATASET_ROOT = "/kaggle/working/unified_dataset_one_class"
DATA_YAML    = os.path.join(DATASET_ROOT, "data.yaml")

# Verify dataset exists
if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(f"Dataset not found at {DATASET_ROOT}. Please run the merging notebook first.")

# Training hyperparameters
EPOCHS       = 100
BATCH_SIZE   = 16      # Adjust based on GPU memory (16 works for T4)
IMAGE_SIZE   = 640
LEARNING_RATE = 0.001
PATIENCE     = 20      # Early stopping

# Output directory for training logs and final model
OUTPUT_DIR = "/kaggle/working/weapon_training"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Using dataset: {DATASET_ROOT}")
print(f"   data.yaml → {DATA_YAML}")

# %% [code]
# ─── 3. Train YOLOv10s Model (single class)
model = YOLO('yolov10s.pt')   # automatically downloads pre‑trained weights

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    lr0=LEARNING_RATE,
    patience=PATIENCE,
    device=0,                    # use GPU
    workers=8,
    project='weapon_training',
    name='yolov10s_weapon',
    exist_ok=True,
    augment=True,
    single_cls=True,             # force single class training (nc=1)
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5
)

print("✅ Training completed!")

# %% [code]
# ─── 4. Evaluate and Save Best Model ────────────────────────────────────────
best_model_path = Path('weapon_training/yolov10s_weapon/weights/best.pt')
if best_model_path.exists():
    # Evaluate on validation set (embedded in data.yaml)
    val_results = model.val(data=DATA_YAML)
    print(f"\n📊 Validation Results (single class 'weapon'):")
    print(f"   mAP@0.5     : {val_results.box.map50:.4f}")
    print(f"   mAP@0.5:0.95: {val_results.box.map:.4f}")
    print(f"   Precision   : {val_results.box.mp:.4f}")
    print(f"   Recall      : {val_results.box.mr:.4f}")

    # Copy best model to a convenient location
    shutil.copy(best_model_path, '/kaggle/working/weapon_best.pt')
    print("\n✅ Best model saved to /kaggle/working/weapon_best.pt")
else:
    print("❌ Best model not found. Check training logs.")